1. Build Your Personalized Knowledge Base: Take your college roll number. Extract its digits. Build a pandas DataFrame with exactly 6 FAQ entries: 4 fixed entries (given in table below) and other 2 entries constructed from your own roll number digits as follows:

• Take the LAST TWO DIGITS of your roll number. For each digit d, compute category = ["billing", "account", "general"][d % 3]. Invent one realistic question+answer+3 keywords per entry that fits the assigned category.

Output: Print your final 6-row DataFrame.


In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

roll_no = 1024160098
last_two_digits = [int(d) for d in str(roll_no)[-2:]]
categories = ["billing", "account", "general"]

print("Roll number:", roll_no)
print("Last two digits:", last_two_digits)

for d in last_two_digits:
    print("digit", d, "-> category", categories[d % 3])


Roll number: 1024160098
Last two digits: [9, 8]
digit 9 -> category billing
digit 8 -> category general


In [2]:
fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    },
]

personalized_entries = [
    {
        "question": "how do i get a refund for overpayment",
        "answer": "Refunds are processed within 7 working days to the original payment method.",
        "keywords": "refund overpayment return",
        "category": "billing"
    },
    {
        "question": "where is the accounts office located",
        "answer": "The accounts office is in Block A, ground floor, near the Hisar campus gate.",
        "keywords": "office location block address",
        "category": "general"
    },
]

df = pd.DataFrame(fixed_entries + personalized_entries)
print(df)


                                question                                                                        answer                       keywords category
0                 what is the annual fee                                                     The annual fee is Rs 500.          fee cost price charge  billing
1                  how to reset password                                              Go to Settings > Reset Password.           password reset login  account
2            what are your working hours                                                     We are open 9 AM to 5 PM.         hours timing open time  general
3                  how can i pay the fee                                    You can pay via UPI, card, or net banking.            pay payment upi fee  billing
4  how do i get a refund for overpayment   Refunds are processed within 7 working days to the original payment method.      refund overpayment return  billing
5   where is the accounts office located  The 

2. Generate and Score a Hypothesis - Implement a scoring function that takes a query string and returns all matching entries ranked by confidence.


In [3]:
def score_query(query, df):
    stopwords = {"how", "to", "do", "i", "a", "the", "is", "my", "can", "what", "are", "your"}
    query_words = [w for w in query.lower().split() if w not in stopwords]
    scores = []

    for _, row in df.iterrows():
        text_words = set((row["question"] + " " + row["keywords"]).lower().split())
        score = sum(1 for word in query_words if word in text_words)
        scores.append(score)

    result = df.copy()
    result["confidence"] = scores
    result = result[result["confidence"] > 0]
    result = result.sort_values("confidence", ascending=False)
    return result


In [4]:
query = "how to reset my password"
print("Query:", query)
print()
print(score_query(query, df))


Query: how to reset my password

                question                            answer              keywords category  confidence
1  how to reset password  Go to Settings > Reset Password.  password reset login  account           2


3. Write a function same_category(category_name, df) that returns all questions belonging to a given category. Call it using the category of one of the personalized entries from Q1, and print the result.


In [5]:
def same_category(category_name, df):
    return df[df["category"] == category_name]


print("Personalized category used: billing")
print()
print(same_category("billing", df))


Personalized category used: billing

                                question                                                                       answer                   keywords category
0                 what is the annual fee                                                    The annual fee is Rs 500.      fee cost price charge  billing
3                  how can i pay the fee                                   You can pay via UPI, card, or net banking.        pay payment upi fee  billing
4  how do i get a refund for overpayment  Refunds are processed within 7 working days to the original payment method.  refund overpayment return  billing


4. Pick any one entry in your knowledge base. Ask the user to input a new keyword, add it to that entry's keywords, and save your entire updated DataFrame to a CSV file named <your_roll_number>_faq_data.csv.


In [6]:
new_keyword = input("Enter a new keyword: ")

df.loc[1, "keywords"] = df.loc[1, "keywords"] + " " + new_keyword

print()
print("Updated entry:")
print(df.loc[1])

df.to_csv("1024160098_faq_data.csv", index=False)
print()
print("Saved as 1024160098_faq_data.csv")


Enter a new keyword: otp

Updated entry:
question               how to reset password
answer      Go to Settings > Reset Password.
keywords            password reset login otp
category                             account
Name: 1, dtype: str

Saved as 1024160098_faq_data.csv


5. Using groupby, print how many FAQ entries you have per category.


In [7]:
print(df.groupby("category").size())


category
account    1
billing    3
general    2
dtype: int64


6. Modify your Q2 scoring function so that if two or more entries tie for the highest score, it does not silently pick one — it prints all matching entries instead, so the user can see every equally good match. Demonstrate with one query that produces a tie (e.g. a query matching both "fee" entries) and one that doesn't.


In [8]:
def score_query_with_ties(query, df):
    result = score_query(query, df)

    if result.empty:
        print("No matching entries found.")
        return result

    highest = result["confidence"].max()
    top = result[result["confidence"] == highest]

    if len(top) > 1:
        print("Multiple entries tied for the highest score:")
    else:
        print("Best matching entry:")

    print(top)
    return top


In [9]:
print("Query that produces a tie:")
print()
score_query_with_ties("fee", df)


Query that produces a tie:

Multiple entries tied for the highest score:
                 question                                      answer               keywords category  confidence
0  what is the annual fee                   The annual fee is Rs 500.  fee cost price charge  billing           1
3   how can i pay the fee  You can pay via UPI, card, or net banking.    pay payment upi fee  billing           1


In [10]:
print("Query that does not produce a tie:")
print()
score_query_with_ties("password", df)


Query that does not produce a tie:

Best matching entry:
                question                            answer                  keywords category  confidence
1  how to reset password  Go to Settings > Reset Password.  password reset login otp  account           1
